# Phase 1: Self-Supervised Pre-Training (SSL) - Corponi Model (Hosseini Dataset)

## Overview
This notebook implements the Phase 1 of the training pipeline, focusing on **Self-Supervised Learning (SSL)**. The objective is to pre-train the feature encoder using a large-scale unlabeled dataset (127 subjects) to learn robust physiological representations.

### Pretext Task: Masked Time-Series Prediction
The model learns by reconstructing masked portions of BVP, EDA, and TEMP signals. This process enables the encoder to capture intrinsic temporal correlations before the supervised fine-tuning phase.

In [ ]:
# ── CELL 1: Environment Configuration ───────────────────────────────────────
import os, sys, shutil

WORK_DIR = '/kaggle/working/e4selflearning'
os.chdir('/kaggle/working')

if os.path.exists(WORK_DIR):
    shutil.rmtree(WORK_DIR)

print("Cloning repository...")
!git clone https://github.com/april-tools/e4selflearning.git {WORK_DIR}

print("Installing core dependencies...")
!pip uninstall -y -q numpy scipy pandas
!pip install --no-cache-dir -q \"numpy==1.26.4\" \"scipy==1.12.0\" \"pandas<2.2\" biopsykit flirt ruamel.yaml biosppy hrv-analysis neurokit2 gdown h5py tqdm peakutils lightning torchmetrics distinctipy joypy

print("\nRestarting kernel to apply library updates...")
import os
os._exit(0)

### 2. Dataset Loading and Path Alignment
In this step, we locate the `provaforsedef` unlabeled dataset and establish symbolic links to the local workspace while ensuring metadata paths are correctly remapped.

In [ ]:
import os, glob, shutil, pickle, numpy as np
os.chdir('/kaggle/working/e4selflearning')

RAW_UNLABELLED = 'data/preprocessed/unsegmented_unlabelled'
os.makedirs(RAW_UNLABELLED, exist_ok=True)

print("Searching for 'provaforsedef' dataset...")
meta_files = glob.glob('/kaggle/input/**/provaforsedef/metadata.pkl', recursive=True)
if not meta_files: meta_files = glob.glob('/kaggle/input/**/metadata.pkl', recursive=True)
if not meta_files: raise FileNotFoundError("CRITICAL ERROR: metadata.pkl not found in input datasets.")

src_dir = os.path.dirname(meta_files[0])
print(f"Source identified: {src_dir}")

for item in os.listdir(src_dir):
    s, d = os.path.join(src_dir, item), os.path.join(RAW_UNLABELLED, item)
    if item == 'metadata.pkl': shutil.copy2(s, d)
    else:
        if os.path.exists(d): os.remove(d)
        os.symlink(s, d)

with open(os.path.join(RAW_UNLABELLED, 'metadata.pkl'), 'rb') as f: meta = pickle.load(f)
new_sessions_info = {}
found_count = 0

for root, dirs, files in os.walk(RAW_UNLABELLED, followlinks=True):
    if 'channels.h5' in files:
        rel_path = os.path.relpath(root, RAW_UNLABELLED).replace('\\', '/')
        for old_id in meta['sessions_info'].keys():
            if old_id.endswith(rel_path) or rel_path.endswith(old_id):
                new_sessions_info[rel_path] = meta['sessions_info'][old_id]
                found_count += 1
                break

meta['sessions_info'] = new_sessions_info
with open(os.path.join(RAW_UNLABELLED, 'metadata.pkl'), 'wb') as f: pickle.dump(meta, f)
print(f"Mapping complete. Valid sessions identified: {found_count}")

### 3. Unlabeled Data Segmentation
The raw signals are partitioned into 512-sample windows with 50% overlap (256-sample step) to prepare for the SSL reconstruction task.

In [ ]:
SEG_UNLABELLED = 'data/preprocessed/sl512_ss256_unlabelled'
print("Starting segmentation process...")
!PYTHONPATH=src python segment.py --data_dir {RAW_UNLABELLED} --output_dir {SEG_UNLABELLED} --segment_length 512 --segmentation_mode 1 --step_size 256 --num_workers 2 --overwrite
print("Segmentation completed successfully.")

### 4. Self-Supervised Training Execution
Running the pre-training loop for 50 epochs using the Masked Prediction pretext task.

In [ ]:
EPOCHS = 50
OUTPUT_DIR = 'output_pretrain'
print("Launching SSL Pre-Training...")
!PYTHONPATH=src python pre_train.py --dataset {SEG_UNLABELLED} --pretext_task masked_prediction --output_dir {OUTPUT_DIR} --epochs {EPOCHS} --batch_size 32 --e4selflearning --verbose 1
print(f"\nTraining complete. Model weights persisted at: {OUTPUT_DIR}/ckpt_sslearner/model_state.pt")